# DataLens agent on Colab

This notebook runs the DataLens text-to-SQL agent on open models with vLLM, next to the plain zero-shot
baseline on the same model, and packages the results for analysis.

**Round 1** (this notebook as it stands) compares baseline and agent on a fixed tuning set of 300
BIRD *train* questions, whose databases never appear in BIRD dev, for Qwen3-Coder-30B-A3B and
Qwen2.5-Coder-32B. It also scores the untuned zero-shot baselines on BIRD dev, which are the "before"
numbers. The agent runs on BIRD dev only once, at the end, with the settings chosen on train.

**Runtime:** a G4 (RTX PRO 6000, 96 GB) or an A100 80GB / H100 (Runtime > Change runtime type).
Then *Runtime > Run all*. Expect roughly 3 to 4 hours, most of it the agent runs; keep the tab open.

**If the runtime disconnects,** run all cells again. Finished questions are skipped, so each run
continues where it stopped. Results live on Google Drive.

**When it finishes,** the last cell saves `datalens/round1-results.zip` to your Drive and downloads it.
Upload that file to the project thread.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## Setup
The install can end with pip reporting "dependency conflicts" for packages Colab preinstalls (such as
RAPIDS or google-adk). Nothing here uses them, so the warning is harmless; the check cell confirms
vLLM works.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

WORKDIR = "/content/drive/MyDrive/datalens"   # persistent outputs
BRANCH = "claude/project-thread-fx3hio"        # the branch with the agent

In [ ]:
%cd /content
!test -d DataLens || git clone -q -b {BRANCH} https://github.com/RohanSada/DataLens.git
%cd /content/DataLens
!git fetch -q origin {BRANCH} && git checkout -q {BRANCH} && git reset -q --hard origin/{BRANCH} && git log --oneline -1
!pip install -q -e ".[eval,vllm]" "transformers==5.13.1"
!mkdir -p {WORKDIR}/runs {WORKDIR}/reports {WORKDIR}/cache
# Runs, reports and caches (schemas, value indexes, gold checks) go to Drive and survive a disconnect.
!ln -sfn {WORKDIR}/runs runs && ln -sfn {WORKDIR}/reports reports && ln -sfn {WORKDIR}/cache .cache

In [ ]:
# Should print GPU: True
!python -c "import torch, vllm; print('GPU:', torch.cuda.is_available(), '| torch', torch.__version__, '| vllm', vllm.__version__)"

## 1. Data
BIRD goes on the runtime's local disk (train is over 30 GB unpacked). A split already in place is skipped.

In [ ]:
!datalens data download --split dev --split train

## 2. Smoke test (about 10 minutes)
Twelve train questions, four agent attempts each, end to end: model load, agent, scoring. If this cell
fails, stop here and send the error.

In [ ]:
!datalens eval suite configs/suites/smoke.yaml
!cat runs/smoke-agent-qwen3-coder/metrics.json | head -40

## 3. Round 1
Each model loads once for all of its runs. The suite logs one `EX` line per run as it scores them.
A model that fails to load is skipped and reported at the end; the other model still runs.

In [ ]:
!datalens eval suite configs/suites/round1.yaml

In [ ]:
from IPython.display import Markdown, display
for report in ["reports/round1-tune/results.md", "reports/round1-dev-baselines/results.md"]:
    try:
        display(Markdown(open(report).read()))
    except FileNotFoundError:
        print("missing:", report)

## 4. Package the results
Bundles each run's config, generations, scores, review file and agent traces, plus the reports.
Upload `round1-results.zip` to the project thread.

In [ ]:
import glob, os, zipfile
out = f"{WORKDIR}/round1-results.zip"
keep = ("config.yaml", "generations.jsonl", "scored.jsonl", "metrics.json", "review.jsonl", "traces.jsonl")
with zipfile.ZipFile(out, "w", zipfile.ZIP_DEFLATED) as zf:
    for path in sorted(glob.glob("runs/*/*")):
        if os.path.basename(path) in keep and not path.startswith("runs/smoke"):
            zf.write(path)
    for path in sorted(glob.glob("reports/round1-*/**", recursive=True)):
        if os.path.isfile(path):
            zf.write(path)
print(out, round(os.path.getsize(out) / 1e6, 1), "MB")
from google.colab import files
files.download(out)